# 全部浓相链网络

指标和路径分布均读取计算产物；网络快照仅从公共链对接触数据派生布局。路径阴影为块SD，布局不是物理距离。


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the project among the working directory and its parents.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulations.yaml").is_file()), None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project, or set ROOT explicitly")
sys.path.insert(0, str(ROOT))
import project
import result_files as files

CONFIG = ROOT / "simulations.yaml"
SELECTED = None  # None means all registered simulations.
OUTPUT = ROOT / "02_calculate/03_contacts/02_dense_chains/02_intermolecular/output"
SIMULATIONS = project.select_simulations(CONFIG, SELECTED)
RESULTS = {s["id"]: files.completed(OUTPUT / s["id"]) for s in SIMULATIONS}
SOURCES = {key: files.read_json(path / "metadata.json") for key, path in RESULTS.items()}
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(RESULTS)}
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
    }
)


def export(fig, name):
    for key, path in RESULTS.items():
        files.completed(path)
        if files.read_json(path / "metadata.json") != SOURCES[key]:
            raise ValueError("Results changed; restart and rerun the notebook")
    for extension in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{extension}", dpi=300, bbox_inches="tight")
    files.write_json(
        FIGURES / f"{name}.sources.json",
        {key: files.read_json(path / "metadata.json") for key, path in RESULTS.items()},
    )


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
for name, path in RESULTS.items():
    summary = pd.read_csv(path / "network/summary.csv").sort_values("threshold")
    for ax, metric in zip(axes, ("global_efficiency", "largest_component_fraction")):
        x = summary.threshold.to_numpy()
        y = summary[f"{metric}_mean_frames"].to_numpy()
        sd = summary[f"{metric}_std_blocks"].to_numpy()
        ax.plot(x, y, marker="o", label=name, color=COLORS[name])
        ax.fill_between(
            x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=COLORS[name], alpha=0.15
        )
        ax.set(
            xlabel="Minimum residue contacts defining an edge",
            ylabel=metric.replace("_", " "),
            ylim=(0, 1.05),
        )
axes[0].legend()
export(fig, "network_thresholds")
plt.show()


In [ ]:
THRESHOLDS = sorted(
    set.intersection(*(set(meta["parameters"]["network_thresholds"]) for meta in SOURCES.values()))
)
if not THRESHOLDS:
    raise ValueError("No shared thresholds among selected results")

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

NETWORK = {name: pd.read_csv(path / "network_frames.csv") for name, path in RESULTS.items()}
SNAPSHOT_FRAME = min(374, min(int(data.frame.max()) for data in NETWORK.values()))
SNAPSHOT_THRESHOLDS = [t for t in (5, 20, 35) if t in THRESHOLDS] or THRESHOLDS[:3]


def load_dense_network(name, frame):
    # Read a bounded chunk; dense nodes are explicitly saved, including isolated nodes.
    for chunk in sorted(
        (ROOT / "02_calculate/03_contacts/00_prepare/output" / name / "edges").glob("*.npz")
    ):
        with np.load(chunk) as saved:
            found = np.flatnonzero(saved["frame"] == frame)
            if len(found):
                index = int(found[0])
                dense = saved["dense_chains"][index]
                n = len(dense)
                matrix = np.zeros((n, n), dtype=np.int32)
                i, j = saved["pair_i"], saved["pair_j"]
                matrix[i, j] = saved["contact_counts"][index]
                matrix[j, i] = saved["contact_counts"][index]
                return matrix[np.ix_(dense, dense)], np.flatnonzero(dense)
    raise ValueError(f"Missing frame {frame} for {name}")


def metrics(name, frame, threshold):
    row = NETWORK[name].query("frame == @frame and threshold == @threshold").iloc[0]
    return row.global_efficiency, row.largest_component_fraction


def fixed_network_layout(contact_counts: np.ndarray) -> np.ndarray:
    """Make a fixed, more evenly spaced topology layout (not physical positions)."""
    if len(contact_counts) == 0:
        return np.empty((0, 2))
    if len(contact_counts) == 1:
        return np.zeros((1, 2))
    if len(contact_counts) == 2:
        return np.array([[-0.5, 0.0], [0.5, 0.0]])
    adjacency = (contact_counts >= 1).astype(float)
    degree = adjacency.sum(axis=1)
    inv_sqrt = np.zeros_like(degree)
    inv_sqrt[degree > 0] = 1 / np.sqrt(degree[degree > 0])
    laplacian = np.eye(len(degree)) - (inv_sqrt[:, None] * adjacency * inv_sqrt[None, :])
    _, vectors = np.linalg.eigh(laplacian)
    xy = vectors[:, 1:3].copy()
    xy += np.random.default_rng(0).normal(scale=1e-3, size=xy.shape)
    xy -= xy.mean(axis=0)
    scale = np.max(np.linalg.norm(xy, axis=1))
    if scale > 0:
        xy /= scale
    # Repel overlapping nodes while keeping connected nodes relatively close.
    target = np.sqrt(4 / len(xy))
    for iteration in range(180):
        delta = xy[:, None, :] - xy[None, :, :]
        distance = np.maximum(np.linalg.norm(delta, axis=-1), 1e-4)
        np.fill_diagonal(distance, 1.0)
        force = ((target**2 / distance - adjacency * distance**2 / target) / distance)[
            ..., None
        ] * delta
        displacement = force.sum(axis=1)
        temperature = 0.08 * (1 - iteration / 180) + 0.002
        magnitude = np.maximum(np.linalg.norm(displacement, axis=1), 1e-12)
        xy += displacement / magnitude[:, None] * np.minimum(magnitude, temperature)[:, None]
        xy -= xy.mean(axis=0)
    xy /= max(float(np.max(np.linalg.norm(xy, axis=1))), 1e-12)
    return xy


## 最短路径长度分布：两条链需要几步连接？

1步是直接接触，2步是经过一条中间链，以此类推。每帧仅统计不同链的无序对，包含孤立链。
分母为该帧全部浓相链对数 N(N−1)/2；不连通对单列。因此有限步数概率之和 + 不连通比例 = 1。
每帧先归一化，再对有效帧等权平均；N<2 的帧缺失。阴影为时间块间 SD，不是置信区间。
全部当前网络阈值均计算，下面 PATH_THRESHOLDS 控制画哪些。横轴为1–10步及>10，长尾概率相加，不连通单列，不重新归一化。


In [ ]:
paths = project.load_module(
    "path_display", ROOT / "02_calculate/03_contacts/dependencies/path_distribution.py"
)
PATH_THRESHOLDS = THRESHOLDS
PATH_MAX = 10  # Long paths are pooled into >10; unreachable pairs remain separate.
path_blocks = pd.concat(
    [pd.read_csv(path / "path_distribution_blocks.csv") for path in RESULTS.values()],
    ignore_index=True,
)
path_display = paths.display_bins(path_blocks, PATH_MAX)


In [ ]:
ncols = min(3, len(PATH_THRESHOLDS))
nrows = int(np.ceil(len(PATH_THRESHOLDS) / ncols))
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 3.3 * nrows),
    squeeze=False,
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
finite = path_display[path_display.bin.gt(0)]
upper = (finite["mean"] + finite.std_blocks.fillna(0)).dropna()
ymax = min(1.02, max(0.05, float(upper.max()) * 1.1)) if len(upper) else 1.02
for ax, threshold in zip(axes.ravel(), PATH_THRESHOLDS):
    for name in RESULTS:
        group = finite[finite.simulation_id.eq(name) & finite.threshold.eq(threshold)].sort_values(
            "bin"
        )
        x, y, sd = group.bin.to_numpy(), group["mean"].to_numpy(), group.std_blocks.to_numpy()
        ax.plot(x, y, marker="o", ms=3, color=COLORS[name], label=name)
        ax.fill_between(
            x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=COLORS[name], alpha=0.15
        )
    ax.set(
        xticks=range(1, PATH_MAX + 2),
        xticklabels=[*map(str, range(1, PATH_MAX + 1)), f">{PATH_MAX}"],
        xlim=(0.5, PATH_MAX + 1.5),
        ylim=(0, ymax),
        xlabel="Shortest path length (edges)",
        ylabel="Fraction of all dense-chain pairs",
        title=f"Edge threshold ≥ {threshold} contacts",
    )
for ax in axes.ravel()[len(PATH_THRESHOLDS) :]:
    ax.set_visible(False)
axes.ravel()[0].legend()
fig.suptitle("Frame-equal mean ± block SD", fontsize=11)
export(fig, "network_path_distribution")
plt.show()
unreachable = path_display[path_display.bin.eq(0)]
fig, ax = plt.subplots(figsize=(6.5, 3.5), constrained_layout=True)
for name in RESULTS:
    group = unreachable[
        unreachable.simulation_id.eq(name) & unreachable.threshold.isin(PATH_THRESHOLDS)
    ].sort_values("threshold")
    x, y, sd = group.threshold.to_numpy(), group["mean"].to_numpy(), group.std_blocks.to_numpy()
    ax.plot(x, y, marker="o", color=COLORS[name], label=name)
    ax.fill_between(x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=COLORS[name], alpha=0.15)
upper = (unreachable["mean"] + unreachable.std_blocks.fillna(0)).dropna()
ax.set(
    xticks=PATH_THRESHOLDS,
    xlabel="Minimum residue contacts defining an edge",
    ylabel="Unreachable pair fraction",
    ylim=(0, min(1.02, max(0.05, float(upper.max()) * 1.1))) if len(upper) else (0, 1.02),
)
ax.legend()
export(fig, "network_unreachable_pairs")
plt.show()
path_display.to_csv(FIGURES / "network_path_display_bins.csv", index=False)
display(unreachable)


In [ ]:
from matplotlib.collections import LineCollection

fig, axes = plt.subplots(
    len(RESULTS),
    len(SNAPSHOT_THRESHOLDS),
    figsize=(3.5 * len(SNAPSHOT_THRESHOLDS), 3 * len(RESULTS)),
    squeeze=False,
    constrained_layout=True,
)
for row, name in enumerate(RESULTS):
    counts, chain_ids = load_dense_network(name, SNAPSHOT_FRAME)
    xy = fixed_network_layout(counts)
    for col, threshold in enumerate(SNAPSHOT_THRESHOLDS):
        ax = axes[row, col]
        adjacency = counts >= threshold
        i, j = np.where(np.triu(adjacency, 1))
        segments = np.stack((xy[i], xy[j]), axis=1)
        ax.add_collection(LineCollection(segments, colors=COLORS[name], alpha=0.2, linewidths=0.7))
        _, labels = connected_components(csr_matrix(adjacency), directed=False)
        largest = np.bincount(labels).argmax() if len(labels) else -1
        colors = [COLORS[name] if label == largest else "#bcbfc4" for label in labels]
        ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=24, edgecolors="white", linewidths=0.4)
        efficiency, giant = metrics(name, SNAPSHOT_FRAME, threshold)
        ax.set(
            title=f"{name} · ≥{threshold} contacts\nE={efficiency:.2f} · LCC={giant:.2f}",
            xlim=(-1.1, 1.1),
            ylim=(-1.1, 1.1),
            aspect="equal",
        )
        ax.axis("off")
export(fig, "network_snapshot")
plt.show()


In [ ]:
import plotly.graph_objects as go

states = []
for variant in RESULTS:
    contact_counts, chain_ids = load_dense_network(variant, SNAPSHOT_FRAME)
    xy = fixed_network_layout(contact_counts)
    for threshold in SNAPSHOT_THRESHOLDS:
        adjacency = contact_counts >= threshold
        edge_i, edge_j = np.where(np.triu(adjacency, 1))
        _, labels = connected_components(csr_matrix(adjacency), directed=False)
        largest_label = np.bincount(labels).argmax() if len(labels) else -1
        efficiency, giant = metrics(variant, SNAPSHOT_FRAME, threshold)
        edge_x, edge_y = [], []
        for i, j in zip(edge_i, edge_j):
            edge_x.extend((xy[i, 0], xy[j, 0], None))
            edge_y.extend((xy[i, 1], xy[j, 1], None))
        degree = adjacency.sum(axis=1)
        node_text = [
            f"Chain {chain_id} (0-based)<br>Partners at threshold {threshold}: {degree[i]}"
            f"<br>{'Largest component' if labels[i] == largest_label else 'Other component'}"
            for i, chain_id in enumerate(chain_ids)
        ]
        node_colors = [COLORS[variant] if label == largest_label else "#bcbfc4" for label in labels]
        states.append(
            {
                "label": f"{variant} · threshold ≥ {threshold}",
                "summary": f"{len(edge_i)} edges · E={efficiency:.2f} · LCC={giant:.2f}",
                "edge_x": edge_x,
                "edge_y": edge_y,
                "node_x": xy[:, 0],
                "node_y": xy[:, 1],
                "node_text": node_text,
                "node_colors": node_colors,
            }
        )

interactive_fig = go.Figure()
for state_index, state in enumerate(states):
    shown = state_index == 0
    interactive_fig.add_trace(
        go.Scatter(
            x=state["edge_x"],
            y=state["edge_y"],
            mode="lines",
            line=dict(color="rgba(100, 105, 112, 0.25)", width=0.8),
            hoverinfo="skip",
            showlegend=False,
            visible=shown,
        )
    )
    interactive_fig.add_trace(
        go.Scatter(
            x=state["node_x"],
            y=state["node_y"],
            mode="markers",
            marker=dict(size=9, color=state["node_colors"], line=dict(color="white", width=0.6)),
            text=state["node_text"],
            hovertemplate="%{text}<extra></extra>",
            showlegend=False,
            visible=shown,
        )
    )

buttons = []
for state_index, state in enumerate(states):
    visibility = [trace_index // 2 == state_index for trace_index in range(2 * len(states))]
    buttons.append(
        dict(
            label=state["label"],
            method="update",
            args=[
                {"visible": visibility},
                {
                    "title": f"{state['label']} · frame {SNAPSHOT_FRAME + 1}<br>"
                    f"<sup>{state['summary']}</sup>"
                },
            ],
        )
    )
interactive_fig.update_layout(
    title=f"{states[0]['label']} · frame {SNAPSHOT_FRAME + 1}<br>"
    f"<sup>{states[0]['summary']}</sup>",
    updatemenus=[dict(buttons=buttons, active=0, x=1, xanchor="right", y=1.16, yanchor="top")],
    width=850,
    height=680,
    margin=dict(l=30, r=30, t=125, b=30),
    plot_bgcolor="white",
    hovermode="closest",
    uirevision="fixed-layout",
    xaxis=dict(visible=False, range=[-1.08, 1.08]),
    yaxis=dict(visible=False, range=[-1.08, 1.08], scaleanchor="x", scaleratio=1),
)
interactive_fig.show(config={"scrollZoom": True, "displaylogo": False})
